In [1]:
! pip install torch

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0 -> 25.2
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

/Users/yanhai/Library/Python/3.9/lib/python/site-packages/torch/nn/modules/transformer.py:20: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  device: torch.device = torch.device(torch._C._get_default_device()),  # torch.device('cpu'),


# 固定位置编码

In [2]:
class FixedPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        # 加入dropout 增强泛化
        self.dropout=nn.Dropout(p=dropout)
        pe=torch.zeros(max_len, d_model)#初始化位置编码矩阵，用于存储每个位置的编码
        # 生成位置索引，[0,1,2,...,max_len-1],形状转置为[max_len,1]
        position=torch.arange(0, max_len ,dtype=torch.float).unsqueeze(1)

        # 计算衰减因子，用于生成不同频率的正弦余弦函数
        div_term=torch.exp(torch.arange(0, d_model, 2, dtype=torch.float)
                           *(-math.log(10000.0 / d_model)))
        
        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position * div_term)
        # 调整形状为 [max_len, 1, d_model],方便与batch中的数据广播相加
        pe = pe.unsqueeze(0)
        # 将位置编码注册为非可学习参数（固定不变）
        # register_buffer 是 PyTorch 为 “需要跟随模型生命周期但无需训练” 的张量设计的便捷机制，
        self.register_buffer('pe', pe) #不会被优化器固定

    def forward(self, x):# x是一个batch的输入，维度为[batchsize, len, d_model]
        x = x + self.pe[:, :x.size(1),:]
        return self.dropout(x)


# 单头注意力机制

In [3]:
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model, d_k, d_v, max_len=5000, pos_dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_k = d_k

        self.pos_encoder =FixedPositionalEncoding(d_model, max_len, pos_dropout)
        self.W_q = nn.Linear(d_model, d_k)
        self.W_k = nn.Linear(d_model, d_k)
        self.W_v = nn.Linear(d_model, d_v)
        self.W_o = nn.Linear(d_v, d_model)

    def forward(self, x, mask=None):
        x=self.pos_encoder(x)
        batch_size = x.size(0)
        Q=self.W_q(x)
        K=self.W_k(x)
        V=self.W_v(x)
        scores = torch.matmul(Q, K.transpose(-2,-1))
        scores = scores/math.sqrt(self.d_k)
        if mask is not None:
            scores =scores.masked_fill(mask==0, -1e9)

        attn_weight = F.softmax(scores, dim=-1)
        output=torch.matmul(attn_weight, V)
        output=self.W_o(output)
        return output, attn_weight

# 多头注意力机制 MHA

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, max_len=5000, pos_dropout=0.1):
        super().__init__()
        self.d_model=d_model
        self.num_heads=num_heads
        self.d_k=d_model // num_heads

        self.pos_encoder=FixedPositionalEncoding(max_len=max_len, dropout=pos_dropout)
        self.W_Q=nn.Linear(d_model, d_model)
        self.W_K=nn.Linear(d_model, d_model)
        self.W_V=nn.Linear(d_model, d_model)
        self.W_O=nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        x=self.pos_encoder(x)
        Q=self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        K=self.W_K(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        V=self.W_V(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        scores=torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(1)==0, -1e9)
        attn_weights = F.softmax(scores, dim=-1)
        output=torch.matmul(attn_weights, V)
        # contiguous()确保张量在内存中连续，避免view报错
        output=output.transpose(1,2).contiguous().view(batch_size,seq_len, self.d_model)
        output=self.W_O(output)
        return output, attn_weights
    


# 多头分组注意力机制 MQA （单组KV共享）

In [ ]:
class MQAAttention(nn.Module):
    '''多查询头，但是只有1组K/V头
       典型配置: num_q_heads = H, num_kv_heads = 1'''
    def __init__(self, d_model, num_q_heads, max_len=5000, pos_dropout=0.1):
        super().__init__()
        self.d_model=d_model
        self.num_q_heads=num_q_heads
        self.d_k=self.d_model // self.num_q_heads
        self.d_v=self.d_model // self.num_q_heads

        self.W_Q=nn.Linear(d_model, d_model)
        self.W_K=nn.Linear(d_model, self.d_k)
        self.W_V=nn.Linear(d_model, self.d_v)

        self.W_O=nn.Linear(d_model, d_model)
        self.pos_encoder=FixedPositionalEncoding(max_len, pos_dropout)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        x=self.pos_encoder(x)
        # 需要把Q 拆分多头 从[batch_size, seq_len, d_model]变成[batch_size, seq_len, num_q_heads, d_k]
        # 然后变成[batch_size, num_q_heads, seq_len, d_k]
        Q=self.W_Q(x).view(batch_size, seq_len, self.num_q_heads, self.d_k).transpose(1,2)
        K=self.W_K(x).unsqueeze(1)
        V=self.W_V(x).unsqueeze(1)
        scores=torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores=scores.masked_fill(mask.unsqueeze(1)==0, -1e9)
        attn_weights = F.softmax(scores, dim=-1)
        output=torch.matmul(attn_weights, V)
        output=output.transpose(1,2).contiguous().view(batch_size, seq_len, self.d_model)
        output=self.W_O(output)
        return output, attn_weights

# GQA 分组共享KV

In [ ]:
class GQAAttention(nn.Module):
    def __init__(self, d_model, num_q_heads, num_groups, max_len, pos_dropout):
        super().__init__()
        self.d_model = d_model
        self.num_q_heads = num_q_heads
        self.groups_size = num_groups #组数
        self.heads_per_group = num_q_heads//num_groups
        self.d_k = d_model // num_q_heads

        self.pos_encoder=FixedPositionalEncoding(d_model, max_len, pos_dropout)

        self.W_Q=nn.Linear(d_model, d_model)
        self.W_K=nn.Linear(d_model, self.groups_size * self.d_k)
        self.W_V=nn.Linear(d_model, self.groups_size * self.d_k)
        self.W_O=nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        x=self.pos_encoder(x)
        # 拆分 Q,转化[batch_size, num_q_heads, seq_len, d_k]
        Q=self.W_Q(x).view(batch_size, seq_len, self.num_q_heads, self.d_k).transpose(1,2)
        
        # 拆分K，V，需要注意拆分方法，拆分后的形式[batch_size, group_size, seq_len, d_k]
        K=self.W_K(x).view(batch_size, seq_len, self.groups_size, self.d_k).transpose(1,2)
        V=self.W_V(x).view(batch_size, seq_len, self.groups_size, self.d_k).transpose(1,2)

        # 扩展K、V以匹配每组内的多个头，（每组的kV共享给组内的所有头）
        # 例如：2组，每组4个头，K、V会扩展成[batch_size, group_size, 4(nums_kv_heads), seq_len, d_k]
        K=K.unsqueeze(2).repeat(1, 1, self.heads_per_group, 1, 1)
        V=V.unsqueeze(2).repeat(1, 1, self.heads_per_group, 1, 1)

        K=K.view(batch_size, self.num_q_heads, seq_len, self.d_k)
        V=V.view(batch_size, self.num_q_heads, seq_len, self.d_k)

        scores = torch.matmul(Q, K.transpose(-2,-1))/math.sqrt(self.d_k)
        if mask is not None:
            scores=scores.masked_fill(mask.unsqueeze(1)==0, -1e9)

        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)
        output = output.transpose(1,2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.W_O(output)
        return output, attn_weights

# MLA （KV 压缩再展开）

In [ ]:
class MLAAttention(nn.Module):
    def __init__(self, d_model, num_heads, r_k, r_v):
        super().__init__()
        self.d_model=d_model
        self.num_heads=num_heads
        self.d_k=d_model//num_heads
        self.r_k=r_k
        self.r_v=r_v
        # Q矩阵、标准的多头映射矩阵
        self.W_Q=nn.Linear(d_model, num_heads * self.d_k)
        
        # K/V 的潜在投影，用来压缩， d_model->r_k/r_v
        self.W_K_latent = nn.Linear(d_model, r_k)
        self.W_V_latent = nn.Linear(d_model, r_v)

        # K/V 的潜在投影，用来扩充到原来维度
        self.W_K_expand = nn.Linear(r_k, num_heads * self.d_k, bias=False)
        self.W_V_expand = nn.Linear(r_v, num_heads * self.d_k, bias=False)

        self.W_O = nn.Linear(num_heads * self.d_k, d_model)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ =x.shape
        Q=self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)

        K_latent=self.W_K_latent(x)
        V_latent=self.W_V_latent(x)

        K_expand=self.W_K_expand(K_latent).view(
            batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        V_expand=self.W_K_expand(V_latent).view(
            batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)


        scores = torch.matmul(Q, K_expand.transpose(-2,-1))/math.sqrt(self.d_k)
        if mask is not None:
            scores=scores.masked_fill(mask.unsqueeze(1)==0, -1e9)

        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V_expand)
        output = output.transpose(1,2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.W_O(output)
        return output, attn_weights